# Module 12: Custom Tokenization


## ✂️ Advanced Custom Tokenization

In Module 4, we learned how to tweak the existing spaCy Tokenizer by adding special cases or modifying regex prefixes/suffixes. 

But what if your text is completely non-standard? What if you are parsing raw code, genetic sequences, or tightly-packed serial numbers? 

In this module, we will learn how to **completely replace** the spaCy tokenizer with our own custom Python class!


<br><br>

---

<br><br>


## 🚀 Building a Custom Tokenizer from Scratch

The tokenizer is the very first step in the pipeline. It takes a raw string of text and returns a `Doc` object.
If you want to build your own tokenizer, you just need a function or class that takes a `str` and returns a `Doc`.

Let's build a simple tokenizer that splits text **only** on whitespace. It won't strip punctuation or do anything fancy.


In [1]:
import spacy
from spacy.tokens import Doc

class WhitespaceTokenizer:
    def __init__(self, vocab):
        self.vocab = vocab

    def __call__(self, text):
        # 1. Split the text on whitespace
        words = text.split(" ")
        
        # 2. To create a Doc, we must also provide an array of booleans indicating
        # whether each word is followed by a space.
        # Since we split on spaces, every word (except the last one) was followed by a space.
        spaces = [True] * len(words)
        spaces[-1] = False # The last word does not have a trailing space
        
        # 3. Create and return the Doc object manually
        return Doc(self.vocab, words=words, spaces=spaces)

nlp = spacy.blank("en")

# Replace the pipeline's default tokenizer with our new one
nlp.tokenizer = WhitespaceTokenizer(nlp.vocab)

# Test it against standard tokenization
doc = nlp("Hello, world! This is a test.")

print("Custom Whitespace Tokens:")
for token in doc:
    print(f"[{token.text}]")
    
print("\nNotice how 'Hello,' and 'world!' keep their punctuation attached!")


Custom Whitespace Tokens:
[Hello,]
[world!]
[This]
[is]
[a]
[test.]

Notice how 'Hello,' and 'world!' keep their punctuation attached!


<br><br>

---

<br><br>


## 🧩 Using Regular Expressions to Tokenize

If whitespace isn't enough, you can use Python's `re` module to find tokens. For example, let's say we want to tokenize a string of Twitter handles or hashtags where spaces don't exist.


In [2]:
import re
from spacy.tokens import Doc

class RegexTokenizer:
    def __init__(self, vocab, pattern):
        self.vocab = vocab
        # Compile the regex pattern
        self.pattern = re.compile(pattern)

    def __call__(self, text):
        # Find all matches in the text
        words = [match.group(0) for match in self.pattern.finditer(text)]
        
        # We assume no spaces between these specific tokens for this example
        spaces = [False] * len(words)
        
        return Doc(self.vocab, words=words, spaces=spaces)

nlp_regex = spacy.blank("en")

# A regex that matches alphanumeric chunks or punctuation
nlp_regex.tokenizer = RegexTokenizer(nlp_regex.vocab, r'\w+|[^\w\s]+')

doc2 = nlp_regex("username@domain.com")
print("Regex Tokens:")
for token in doc2:
    print(f"[{token.text}]")


Regex Tokens:
[username]
[@]
[domain]
[.]
[com]


<br><br>

---

<br><br>


## ✂️ Modifying Tokens Post-Creation (Retokenization)

Sometimes, it's easier to let the default tokenizer do its job, and then fix the mistakes afterward using a custom pipeline component.

You can use `doc.retokenize()` to either **merge** multiple tokens into one, or **split** one token into many!

### Merging Tokens


In [3]:
import spacy
from spacy.language import Language

@Language.component("merge_custom_acronyms")
def merge_acronyms(doc):
    # Let's say we want "N.Y.C." to always be one token.
    # The default tokenizer sometimes splits on punctuation if not configured.
    
    with doc.retokenize() as retokenizer:
        # We will loop through looking for consecutive N, ., Y, ., C, .
        # (For simplicity, we'll just merge the first 6 tokens if it matches)
        text_span = doc[0:6].text
        if text_span == "N.Y.C.":
            retokenizer.merge(doc[0:6])
            
    return doc

# Create a blank model, which uses the default tokenizer
nlp = spacy.blank("en")
nlp.add_pipe("merge_custom_acronyms")

doc3 = nlp("N.Y.C. is a large city.")
print(f"Merged token: [{doc3[0].text}]")


Merged token: [N.Y.C.]


<br><br>

---

<br><br>


### Splitting Tokens

You can also split a single token into multiple tokens. This requires you to specify the new text for each split, and which component of the split should be considered the "head" (for the dependency parser).


In [4]:
@Language.component("split_custom_words")
def split_words(doc):
    with doc.retokenize() as retokenizer:
        for token in doc:
            if token.text == "alot":
                # We want to split 'alot' into 'a' and 'lot'
                # heads specify which token is the parent (index 1 means 'lot' is the parent of 'a')
                retokenizer.split(token, ["a", "lot"], heads=[(token, 1), (token, 0)])
    return doc

nlp.add_pipe("split_custom_words")

doc4 = nlp("I like this alot.")
print("Split tokens:")
for token in doc4:
    print(f"[{token.text}]")


Split tokens:
[I]
[like]
[this]
[a]
[lot]
[.]


<br><br>

---

<br><br>


## 🎉 Summary of Part 4

Congratulations! You have completed **Part 4: Customization & Extension**.

You are now a spaCy power user. You know how to:
- Write your own custom pipeline components to run logic automatically.
- Add custom extension attributes to permanently save data on the `Doc`.
- Completely rewrite the internal tokenizer to handle wildly custom data formats.

In **Part 5**, we enter the world of Machine Learning! We will learn how to take a blank spaCy model and **train it from scratch** to recognize brand new entity types and text categories using real training data.
